# LU Row

No pivoting. Right-looking elimination uses i then j (row updates) after computing each multiplier column. PyTorch batches the innermost loop.

Run from the repository folder or any folder below it. Requires NumPy, SciPy and Numba.


In [1]:
import time
import sys
from pathlib import Path
import numpy as np
from scipy.sparse import csr_matrix, issparse

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start the notebook inside Numerical_Linear_Algebra_Algorithms")
sys.path.insert(0, str(ROOT))
from common import loader, data_path, lu_solve, relative_residual
from direct_methods.lu_row import lu_row


def timed(fn, *args):
    start = time.perf_counter()
    result = fn(*args)
    return result, time.perf_counter() - start


## 3×3


In [2]:
A = np.array([[4., 1., 0.], [1., 3., 1.], [0., 1., 2.]])
x_true = np.ones(3)
b = A @ x_true
rng = np.random.default_rng(42)
Z = rng.normal(size=(A.shape[1], 3))
_ = lu_row(A)
result, seconds = timed(lu_row, A)
print("Seconds:", seconds)
L, U = result
x = lu_solve(L, U, b)
print("Factorization probe error:", float(np.linalg.norm(A @ Z - L @ (U @ Z)) / np.linalg.norm(A @ Z)))
print("Relative residual:", relative_residual(A, x, b))
print("Relative solution error:", float(np.linalg.norm(x-x_true) / np.linalg.norm(x_true)))
assert relative_residual(A, x, b) < 1e-8


Seconds: 4.375004209578037e-06
Factorization probe error: 3.93419176300259e-17
Relative residual: 0.0
Relative solution error: 0.0


## SuiteSparse

Change MATRIX to an existing file. bcsstk09 is 1,083×1,083; bcsstk17 is 10,974×10,974; west1505 is nonsymmetric and requires pivoting. No 10^5 matrix exists in this repo. Dense factorization costs O(n³); run the optional 10^4 case deliberately.


In [3]:
MATRIX = "bcsstk09"
MAT_PATH = None
A = loader("suitesparse", path=data_path(MATRIX, MAT_PATH), dense=True)
print("Matrix:", MATRIX, "shape:", A.shape)
x_true = np.ones(A.shape[0])
b = A @ x_true
rng = np.random.default_rng(42)
Z = rng.normal(size=(A.shape[1], 3))


Matrix: bcsstk09 shape: (1083, 1083)


In [4]:
result, seconds = timed(lu_row, A)
print("Seconds:", seconds)
L, U = result
x = lu_solve(L, U, b)
print("Factorization probe error:", float(np.linalg.norm(A @ Z - L @ (U @ Z)) / np.linalg.norm(A @ Z)))
print("Relative residual:", relative_residual(A, x, b))
print("Relative solution error:", float(np.linalg.norm(x-x_true) / np.linalg.norm(x_true)))


Seconds: 0.4984612910193391
Factorization probe error: 3.8806135147087197e-16
Relative residual: 8.155022779321071e-16
Relative solution error: 1.5372172278821023e-13


In [5]:
RUN_10_4 = False

if RUN_10_4:
    MATRIX = "bcsstk17"
    MAT_PATH = None
    A = loader("suitesparse", path=data_path(MATRIX, MAT_PATH), dense=True)
    print("Matrix:", MATRIX, "shape:", A.shape)
    x_true = np.ones(A.shape[0])
    b = A @ x_true
    rng = np.random.default_rng(42)
    Z = rng.normal(size=(A.shape[1], 3))
    result, seconds = timed(lu_row, A)
    print("Seconds:", seconds)
    L, U = result
    x = lu_solve(L, U, b)
    print("Factorization probe error:", float(np.linalg.norm(A @ Z - L @ (U @ Z)) / np.linalg.norm(A @ Z)))
    print("Relative residual:", relative_residual(A, x, b))
    print("Relative solution error:", float(np.linalg.norm(x-x_true) / np.linalg.norm(x_true)))
